# Error Not Rerun analysis

Parse a `file_status_<model>.txt` report to list `Error Not Rerun` entries, summarize counts, and group by inferred category (prefix of the ID before the final underscore).

In [10]:
from pathlib import Path
import pandas as pd
import re

# Point to the file_status report you want to analyze
REPORT_PATH = Path("<PROJECT_ROOT>/Desktop/Desktop - ADUAED19365LPMX/Agent_IX_Personalization/gorilla/berkeley-function-call-leaderboard/scripts/file_status_claude-opus-4-5-20251101-FC.txt")

text = REPORT_PATH.read_text(encoding="utf-8")
lines = text.splitlines()

records = []
current_variant = None
current_persona = None
current_bucket = None

for line in lines:
    stripped = line.strip()
    if stripped.startswith("==") and stripped.endswith("=="):
        current_variant = stripped.strip("= ")
        continue
    if stripped.startswith("- ") and stripped.endswith(":"):
        current_persona = stripped[2:-1]
        current_bucket = None
        continue
    # Bucket line (indented, ends with colon)
    if stripped.endswith(":") and not stripped.startswith("- "):
        current_bucket = stripped[:-1]
        continue
    if current_bucket != "Error Not Rerun":
        continue
    if stripped.startswith("-"):
        # Entry line under Error Not Rerun
        entry_text = stripped[1:].strip()
        entry_id = entry_text.split()[0]
        category = entry_id.rsplit("_", 1)[0] if "_" in entry_id else entry_id
        records.append(
            {
                "variant": current_variant,
                "persona": current_persona,
                "entry_id": entry_id,
                "category": category,
                "detail": entry_text,
            }
        )

df = pd.DataFrame(records)


In [11]:
df.head()

,variant,persona,entry_id,category,detail
0,no_personalization,tool_high,multi_turn_long_context_157,multi_turn_long_context,multi_turn_long_context_157 (log_lines=94)
1,no_personalization,param_low,multi_turn_long_context_10,multi_turn_long_context,multi_turn_long_context_10 (log_lines=88)
2,no_personalization,param_low,multi_turn_long_context_13,multi_turn_long_context,multi_turn_long_context_13 (log_lines=81)
3,no_personalization,param_low,multi_turn_long_context_163,multi_turn_long_context,multi_turn_long_context_163 (log_lines=91)
4,no_personalization,param_medium,multi_turn_long_context_163,multi_turn_long_context,multi_turn_long_context_163 (log_lines=88)


In [7]:
# Summary: counts by entry_id (具体哪个sample出现的最多)
if df.empty:
    print("No 'Error Not Rerun' entries found.")
else:
    # Find most frequent samples (entry_id frequency)
    sample_counts = (
        df["entry_id"]
        .value_counts()
        .reset_index()
        .rename(columns={"index": "entry_id", "entry_id": "count"})
    )
    display(sample_counts)

    # 也可以显示每个entry_id所属的category，按count降序
    sample_with_cat = (
        df.groupby(["entry_id", "category"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    display(sample_with_cat)

    # Optionally: full listing with variant/persona context
    display(df.sort_values(["variant", "persona", "entry_id"]).reset_index(drop=True))


,count,count
0,multi_turn_miss_param_2,3
1,multi_turn_miss_param_100,2
2,multi_turn_long_context_199,1
3,multi_turn_long_context_9,1
4,multi_turn_long_context_163,1
5,multi_turn_miss_param_0,1
6,multi_turn_long_context_150,1


,entry_id,category,count
6,multi_turn_miss_param_2,multi_turn_miss_param,3
5,multi_turn_miss_param_100,multi_turn_miss_param,2
0,multi_turn_long_context_150,multi_turn_long_context,1
1,multi_turn_long_context_163,multi_turn_long_context,1
2,multi_turn_long_context_199,multi_turn_long_context,1
3,multi_turn_long_context_9,multi_turn_long_context,1
4,multi_turn_miss_param_0,multi_turn_miss_param,1


,variant,persona,entry_id,category,detail
0,no_personalization,error_discovery_detail,multi_turn_long_context_199,multi_turn_long_context,multi_turn_long_context_199 (log_lines=81)
1,no_personalization,info_collect_upfront,multi_turn_miss_param_2,multi_turn_miss_param,multi_turn_miss_param_2 (log_lines=82)
2,personalization,info_collect_gradual,multi_turn_miss_param_0,multi_turn_miss_param,multi_turn_miss_param_0 (log_lines=84)
3,personalization,info_collect_gradual,multi_turn_miss_param_100,multi_turn_miss_param,multi_turn_miss_param_100 (log_lines=82)
4,personalization,info_collect_gradual,multi_turn_miss_param_2,multi_turn_miss_param,multi_turn_miss_param_2 (log_lines=81)
5,personalization,info_collect_upfront,multi_turn_miss_param_100,multi_turn_miss_param,multi_turn_miss_param_100 (log_lines=81)
6,personalization,info_collect_upfront,multi_turn_miss_param_2,multi_turn_miss_param,multi_turn_miss_param_2 (log_lines=83)
7,personalization,param_medium,multi_turn_long_context_163,multi_turn_long_context,multi_turn_long_context_163 (log_lines=83)
8,personalization,tool_invocation_single,multi_turn_long_context_150,multi_turn_long_context,multi_turn_long_context_150 (log_lines=81)
9,personalization,tool_low,multi_turn_long_context_9,multi_turn_long_context,multi_turn_long_context_9 (log_lines=81)
